# Imports

In [1]:
import time
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Bidirectional, Dense, Dropout

# Data

In [2]:
vocab_size = 10000
max_len = 200
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


# Build models

In [3]:
def build_model(cell_type="RNN", bidirectional=False):
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len))

    if cell_type == "RNN":
        layer = SimpleRNN(64)
    elif cell_type == "LSTM":
        layer = LSTM(64)
    elif cell_type == "GRU":
        layer = GRU(64)

    if bidirectional:
        model.add(Bidirectional(layer))
    else:
        model.add(layer)

    model.add(Dropout(0.5))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Train and Evaluate

In [4]:
models = {
    "RNN": build_model("RNN", False),
    "BiRNN": build_model("RNN", True),
    "LSTM": build_model("LSTM", False),
    "BiLSTM": build_model("LSTM", True),
    "GRU": build_model("GRU", False),
    "BiGRU": build_model("GRU", True),
}

results = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(x_train, y_train, epochs=3, batch_size=64, validation_split=0.2, verbose=1)
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results[name] = acc


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training RNN...
Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.6980 - loss: 0.5731 - val_accuracy: 0.8018 - val_loss: 0.4448
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8256 - loss: 0.3994 - val_accuracy: 0.7728 - val_loss: 0.4850
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9165 - loss: 0.2175 - val_accuracy: 0.8076 - val_loss: 0.5041
Training BiRNN...
Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.6001 - loss: 0.6521 - val_accuracy: 0.7774 - val_loss: 0.4788
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.8037 - loss: 0.4355 - val_accuracy: 0.7640 - val_loss: 0.4934
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.8937 - loss: 0.2629 - val_accuracy: 0.7704 - val_loss: 0.5285
Training LSTM...
Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.7938 - loss: 0.4342 - val_accuracy: 0.8654 - val_loss: 0.3243
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/s

In [5]:
results

{'RNN': 0.7992799878120422,
 'BiRNN': 0.7748000025749207,
 'LSTM': 0.8497200012207031,
 'BiLSTM': 0.8438000082969666,
 'GRU': 0.8655200004577637,
 'BiGRU': 0.8653600215911865}